# MACURA on Drones — Colab Notebook

**The only place code runs.** Clone the repo → install → import the pure-function
modules → study the environment → train the four algorithms → plot.

> **Part 1 (this notebook, ready to run):** environment creation + study-case
> plots.  
> **Part 2:** the four-algorithm comparison (MACURA / MBPO / M2AC / SAC).

Results mirror to Google Drive so they survive a session reset.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Provide the GitHub token (private repo) — SECURELY

The project repo is **private**, so cloning needs a GitHub Personal Access Token
(PAT, `repo` scope). **Do not paste the token into any code cell or commit it.**
Either:
* **Recommended:** store it in Colab **Secrets** (🔑 left sidebar) as `GITHUB_TOKEN`
  — the cell below reads it without ever displaying it; or
* fall back to a hidden `getpass` prompt.

The token is held only in memory for the single `git clone` and never written to disk.

In [ ]:
import os
TOKEN = None
# 1) try Colab Secrets
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
# 2) fall back to a hidden prompt
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded into environment (not displayed).')

## 2. Bootstrap: clone repo + Skydio X2 model + install deps

We download `setup_colab.sh` directly via the GitHub API (so it works even
before the repo is cloned), then run it. It clones this repo, `mujoco_menagerie`
(for the Skydio X2 MJCF), optionally the authors' MACURA code, installs
`requirements.txt`, and creates the Drive results folders.

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=macura-drone
# fetch the setup script from the private repo via the API (uses the token)
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
# make the cloned repo importable
import sys
sys.path.insert(0, '/content/macura-drone')
# headless rendering for MuJoCo
os.environ['MUJOCO_GL'] = 'egl'

## 3. Load the config

In [ ]:
import yaml
with open('/content/macura-drone/configs/macura_drone.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
DRIVE = cfg['experiment']['drive_root']
cfg['experiment']

# PART 1 — Environment study & plots

Create the Skydio X2 hover environment and visualize its behaviour BEFORE any
training. This validates the env (dynamics, reward, instability) and produces
the study figures.

## 4. Create the environment

In [ ]:
from envs import drone_env
from viz import plots

env, obs_dim, act_dim = drone_env.make_env(cfg['env'], seed=0, render=True)
print(f'obs_dim={obs_dim}, act_dim={act_dim}')
print('actuator ctrl ranges:\n', env.model.actuator_ctrlrange)

## 5. Render the drone

In [ ]:
env.reset(seed=0)
plots.render_frame(env);

## 6. Study case A — random policy (shows the platform is unstable)

With random motor commands the drone tumbles fast — this is exactly the
instability that stresses model-based rollouts.

In [ ]:
rec_rand = plots.collect_rollout(env, policy='random', num_steps=250, seed=0)
plots.plot_rollout_traces(rec_rand, title='Random policy',
                          save_path=f'{DRIVE}/plots/study_random_rollout.png');

## 7. Study case B — zero/constant action (free dynamics)

In [ ]:
rec_hover = plots.collect_rollout(env, policy='hover', num_steps=250, seed=0)
plots.plot_rollout_traces(rec_hover, title='Constant (zero) action',
                          save_path=f'{DRIVE}/plots/study_constant_action.png');

## 8. Study case C — state distribution under random policy

In [ ]:
plots.plot_state_distributions(env, num_episodes=20, num_steps=100,
                               save_path=f'{DRIVE}/plots/study_state_dist.png');

## 9. Study case D — reward landscape (smooth & model-friendly)

In [ ]:
plots.plot_reward_landscape(cfg, save_path=f'{DRIVE}/plots/study_reward.png');

## 10. Study case E — action response (thrust → climb sanity check)

In [ ]:
plots.plot_action_response(env, save_path=f'{DRIVE}/plots/study_action_response.png');

---
# PART 2 — Four-algorithm comparison (scaffold)

Train MACURA / MBPO / M2AC / SAC with the SHARED SAC backbone and ensemble,
identical seeds and evaluation. **Compute-heavy** — start with 1 seed / short
steps to smoke-test, then scale up. See `tasks.md` for the bring-up checklist.

In [ ]:
from training import train as trainer

# Smoke test: one short MACURA run (reduce steps first to validate the pipeline)
cfg_smoke = {**cfg}
cfg_smoke['experiment'] = {**cfg['experiment'], 'total_env_steps': 3000,
                           'seeds': [0]}
run = trainer.train_one('macura', cfg_smoke, DRIVE, seed=0)
print('eval returns:', run['eval_return'])

In [ ]:
# Full comparison: loop over algorithms x seeds (long!)
runs = []
for algo in cfg['experiment']['algorithms']:
    for seed in cfg['experiment']['seeds']:
        runs.append(trainer.train_one(algo, cfg, DRIVE, seed=seed))

In [ ]:
# Comparison figures
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/sample_efficiency.png')
plots.plot_failure_rate(runs, save_path=f'{DRIVE}/plots/failure_rate.png')
plots.plot_final_quality(runs, save_path=f'{DRIVE}/plots/final_quality.png')
macura_runs = [r for r in runs if r['algo'] == 'macura']
if macura_runs:
    plots.plot_rollout_depth(macura_runs[0], save_path=f'{DRIVE}/plots/rollout_depth.png')